# EduRecSys – Content-Based Recommendation

This notebook implements a **Content-Based Recommendation System** using textual similarity.
The goal is to recommend educational content based on the semantic similarity of course/article descriptions.

Key features:
- TF-IDF text representation
- Cosine similarity
- Support for English and Arabic content
- Cold-start friendly (no user data required)


In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from pathlib import Path


# Load Preprocessed Data

In [2]:
BASE_PATH = Path("/kaggle/input/edurecsys-processed-content")  

EN_PATH = BASE_PATH / "/kaggle/input/edurecsys-processed-content/content_en.csv"
AR_PATH = BASE_PATH / "/kaggle/input/edurecsys-processed-content/content_ar.csv"

content_en = pd.read_csv(EN_PATH)
content_ar = pd.read_csv(AR_PATH)

print(content_en.shape, content_ar.shape)


(3678, 7) (13000, 7)


In [4]:
content_en.info()

content_en.head()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3678 entries, 0 to 3677
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   content_id        3678 non-null   int64  
 1   title             3678 non-null   object 
 2   description       3678 non-null   object 
 3   subject           3678 non-null   object 
 4   level             3678 non-null   object 
 5   content_duration  3678 non-null   float64
 6   language          3678 non-null   object 
dtypes: float64(1), int64(1), object(5)
memory usage: 201.3+ KB


,content_id,title,description,subject,level,content_duration,language
0,1070968,Ultimate Investment Banking Course,Ultimate Investment Banking Course,Business Finance,All Levels,1.5,en
1,1113822,Complete GST Course & Certification - Grow You...,Complete GST Course & Certification - Grow You...,Business Finance,All Levels,39.0,en
2,1006314,Financial Modeling for Business Analysts and C...,Financial Modeling for Business Analysts and C...,Business Finance,Intermediate Level,2.5,en
3,1210588,Beginner to Pro - Financial Analysis in Excel ...,Beginner to Pro - Financial Analysis in Excel ...,Business Finance,All Levels,3.0,en
4,1011058,How To Maximize Your Profits Trading Options,How To Maximize Your Profits Trading Options,Business Finance,Intermediate Level,2.0,en


In [5]:
content_ar.info()

content_ar.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13000 entries, 0 to 12999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   content_id        13000 non-null  object 
 1   title             13000 non-null  object 
 2   description       13000 non-null  object 
 3   subject           13000 non-null  object 
 4   level             13000 non-null  object 
 5   content_duration  13000 non-null  float64
 6   language          13000 non-null  object 
dtypes: float64(1), object(6)
memory usage: 711.1+ KB


,content_id,title,description,subject,level,content_duration,language
0,Tech_1893,تستعد اتصالات لطرح نظام التحاسب بالثانية لجميع...,تستعد اتصالات لطرح نظام التحاسب بالثانية لجميع...,Tech,Intermediate,2.07,ar
1,Tech_1711,أعلنت وحدة الأعمال المتنقلة التابعة لشركة موتو...,أعلنت وحدة الأعمال المتنقلة التابعة لشركة موتو...,Tech,Intermediate,1.72,ar
2,Tech_4682,تطالب شركة جوجل، صاحبة محرك البحث الشهير على,تطالب شركة جوجل، صاحبة محرك البحث الشهير على ا...,Tech,Beginner,0.91,ar
3,Tech_5450,اختارت شركة الثريا للاتصالات شركة فورت انفو تك...,اختارت شركة الثريا للاتصالات شركة فورت انفو تك...,Tech,Beginner,0.97,ar
4,Tech_5064,أعلنت دو أنها ستبدأ قريباً بدعوة عُملائها لتسجيل,أعلنت دو أنها ستبدأ قريباً بدعوة عُملائها لتسج...,Tech,Beginner,0.71,ar


# Text Preprocessing

In [6]:
def clean_text(text):
    if pd.isna(text):
        return ""
    return text.lower()

content_en["clean_text"] = content_en["description"].apply(clean_text)
content_ar["clean_text"] = content_ar["description"].fillna("")


A minimal text preprocessing step was applied before vectorization.

- Missing text values were handled to avoid errors during TF-IDF computation.
- English text was lowercased to normalize word representations.
- Arabic text was kept unchanged since case normalization is not required.

The preprocessing was intentionally kept simple to maintain model interpretability
and focus on content similarity rather than linguistic complexity.


# TF-IDF Vectorization

In [12]:
tfidf_en = TfidfVectorizer(
    max_features=6000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)
tfidf_matrix_en = tfidf_en.fit_transform(content_en["clean_text"])

In [13]:
tfidf_ar = TfidfVectorizer(
    max_features=6000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)
tfidf_matrix_ar = tfidf_ar.fit_transform(content_ar["clean_text"])


In this step, textual descriptions were transformed into numerical representations using **TF-IDF (Term Frequency–Inverse Document Frequency)**.

- TF-IDF assigns higher weights to terms that are important within a document while reducing the influence of commonly occurring words.
- Both **unigrams and bigrams** were used (`ngram_range=(1, 2)`) to better capture multi-word educational concepts such as *machine learning* and *data science*.
- Vocabulary size was controlled using `max_features`, while rare and overly common terms were filtered using `min_df` and `max_df`.

Separate TF-IDF models were trained for:
- **English content (Udemy courses)**
- **Arabic content (SANAD articles)**

This language-specific vectorization ensures accurate semantic similarity computation for multilingual content.


# Cosine Similarity

After transforming course descriptions into TF-IDF vectors, cosine similarity is used to measure the semantic similarity between content items.

Cosine similarity computes the cosine of the angle between two vectors, producing a similarity score between 0 and 1, where higher values indicate greater similarity.

This similarity matrix serves as the core component of the content-based recommendation system.


In [14]:
similarity_en = cosine_similarity(tfidf_matrix_en)
similarity_ar = cosine_similarity(tfidf_matrix_ar)

# Recommendation Function

In this section, we build a recommendation function that retrieves the most similar content items based on cosine similarity between TF-IDF vectors.

In [16]:
def recommend_en(content_id, top_k=5):
    #Get index of the selected content
    idx = content_en.index[content_en["content_id"] == content_id][0]
    
    # Get similarity scores for this content
    sim_scores = list(enumerate(similarity_en[idx]))

    # Sort by similarity score (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Exclude the item itself
    sim_scores = sim_scores[1: top_k + 1]
    
    # Get indices of recommended items
    rec_indices = [i[0] for i in sim_scores]
    
    return content_en.iloc[rec_indices][
        ["content_id", "title", "subject", "level"]
    ]


In [18]:
recommend_en(content_en.iloc[0]["content_id"])

,content_id,title,subject,level
39,965832,The Complete Investment Banking Course 2017,Business Finance,All Levels
417,736836,The Investment Banking Recruitment Series,Business Finance,All Levels
240,298558,Advanced Accounting for Investment Banking,Business Finance,Intermediate Level
227,511952,Investment Banking: How to Land a Job on Wall ...,Business Finance,All Levels
137,730414,"Intro to Investment Banking, M&A, IPO, Modelin...",Business Finance,All Levels


In [17]:
def recommend_ar(content_id, top_k=5):
    #Get index of the selected content
    idx = content_ar.index[content_ar["content_id"] == content_id][0]
    
    # Get similarity scores for this content
    sim_scores = list(enumerate(similarity_ar[idx]))
    
    # Sort by similarity score (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Exclude the item itself
    sim_scores = sim_scores[1: top_k + 1]

    # Get indices of recommended items
    rec_indices = [i[0] for i in sim_scores]
    return content_ar.iloc[rec_indices][
        ["content_id", "description", "subject", "level"]
    ]


In [19]:
recommend_ar(content_ar.iloc[0]["content_id"])

,content_id,description,subject,level
4939,Tech_5108,استكملت كل من اتصالات ودو الجزء الاكبر من استع...,Tech,Intermediate
4488,Tech_4422,أبوظبي - سامح الليثي:تستعد هيئة تنظيم الاتصالا...,Tech,Intermediate
2850,Tech_2472,مددت هيئة تنظيم الاتصالات المهلة الممنوحة لكل ...,Tech,Intermediate
2652,Tech_5811,اقتربت كل من #187;اتصالات#171; و#187;دو#171; م...,Tech,Intermediate
6062,Tech_1110,يشهد قطاع الاتصالات بالدولة حالياً اتجاه من ات...,Tech,Intermediate


## Recommendation Function Explanation

This function implements a content-based recommendation approach using cosine similarity.

Given a selected content item:
- The corresponding index is retrieved from the dataset.
- Cosine similarity scores between the selected item and all other items are extracted.
- Items are ranked based on similarity scores in descending order.
- The selected item itself is excluded from the recommendations.
- The top-K most similar content items are returned.

Separate recommendation functions are implemented for Arabic and English content to ensure accurate language-specific similarity matching.
